# 07-01 · 필요한 계정이 응답에 '있는지' 세기

| 항목 | 내용 |
|---|---|
| 하는 일 | 본수집 응답 안에 영업이익·금융비용·이자비용·차입 계정 등이 **있는지, 값 칸이 비어 있지 않은지**만 센다 (A8 판단 목록 1·2·3번 재료) |
| 읽는 파일 | `outputs/04_본수집/본수집_로그.csv`, 저장된 응답 `data/raw/DART_파일럿/fs/` |
| 만드는 파일 | `outputs/05_계정확보/계정확보_보고서별.csv`, `계정확보_기업별.csv`, `계정확보_업종별.csv` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 1분 28초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

금액은 읽지 않고 ICR·노출도 E도 계산하지 않습니다. 계정 이름 규칙은 파일럿의 `config.ACCOUNT_PATTERNS`를 **결과를 보고 고치지 않고** 그대로 씁니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""본 수집 응답 안에 필요한 계정이 '있는지'만 센다 (A8 판단 목록의 1·2·3번 재료).

이 스크립트가 하는 일
  1. outputs/04_본수집/본수집_로그.csv에서 상태 000인 응답(기업·연도·보고서·연결/별도)을 읽는다.
     (같은 기업·연도·보고서에 연결이 000이면 연결, 아니면 별도: 본 수집이 연결을 먼저 물었기 때문)
  2. 각 응답 파일에서 config.ACCOUNT_PATTERNS의 계정이 있는지, 값 칸이 비어 있지 않은지만 기록한다.
     금액 자체는 읽거나 계산하지 않는다. ICR·노출도 E는 계산하지 않는다.
  3. 보고서별 표, 기업별 표, 법인구분·업종별 요약을 outputs/05_계정확보/에 저장한다.

계정 이름 규칙은 파일럿에서 쓴 config.ACCOUNT_PATTERNS를 그대로 쓴다(이번 결과를 보고 고치지 않는다.
고치게 되면 이유와 함께 사전 고정 문서 변경 기록에 적는다).

실행: python -X utf8 01_계정_확보현황.py            (저장된 파일만 읽음, 키 필요 없음)
"""
import csv
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
LOG = config.BASE / "outputs" / "04_본수집" / "본수집_로그.csv"
RESULT = config.BASE / "outputs" / "04_본수집" / "본수집_결과_기업별.csv"
OUTDIR = config.BASE / "outputs" / "05_계정확보"
FS_DIR = config.RAW / "fs"
REP_ORDER = {"Q1": 0, "H1": 1, "Q3": 2, "FY": 3}
PATS = {k: (set(v[0].split("|")), re.compile(v[1])) for k, v in config.ACCOUNT_PATTERNS.items()}
KEYS = list(PATS)
# 누적 칸(thstrm_add_amount)이 필요한 손익 계정
ADD_KEYS = ["op_income", "interest_exp", "fin_cost"]

#### `is_pre()` — 사전기간 2019Q1~2021Q2 (10개 보고서)

In [3]:
def is_pre(year, rep):
    """사전기간 2019Q1~2021Q2 (10개 보고서)"""
    return (2019, 0) <= (year, REP_ORDER[rep]) <= (2021, 1)

#### `nonempty()`

In [4]:
def nonempty(v):
    return v is not None and str(v).strip() not in ("", "-")

#### `scan()`

In [5]:
def scan(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = data.get("list", [])
    out = {"n_rows": len(rows)}
    for k, (sjs, pat) in PATS.items():
        hits = [r for r in rows if r.get("sj_div") in sjs and pat.search(re.sub(r"\s+", "", r.get("account_nm", "")))
                and nonempty(r.get("thstrm_amount"))]
        out[k] = bool(hits)
        out[k + "_n"] = len(hits)
    for k in ADD_KEYS:
        sjs, pat = PATS[k]
        out[k + "_누적칸"] = any(r.get("sj_div") in sjs and pat.search(re.sub(r"\s+", "", r.get("account_nm", "")))
                              and nonempty(r.get("thstrm_add_amount")) for r in rows)
    return out

#### `main()`

이 함수(118줄) 안의 단계 — 코드 주석에서 뽑은 목차:
1. 기업별 집계
2. ── 요약 출력 ──

In [6]:
def main():
    firms = {r["corp_code"]: r for r in csv.DictReader(open(RESULT, encoding="utf-8-sig"))}
    chosen = {}                                  # (corp, year, rep) -> fs
    for r in csv.DictReader(open(LOG, encoding="utf-8-sig")):
        if r["status"] != "000":
            continue
        k = (r["corp_code"], int(r["year"]), r["report"])
        if k not in chosen or r["fs_div"] == "CFS":
            chosen[k] = r["fs_div"]
    print(f"확인할 응답 {len(chosen):,}개 (기업 {len(firms):,}곳)")

    OUTDIR.mkdir(parents=True, exist_ok=True)
    per_report = []
    for i, ((corp, year, rep), fs) in enumerate(sorted(chosen.items()), 1):
        p = FS_DIR / f"{corp}_{year}_{config.REPORTS[rep]}_{fs}.json"
        s = scan(p)
        s.update({"corp_code": corp, "year": year, "report": rep, "fs_div": fs, "사전기간": is_pre(year, rep)})
        per_report.append(s)
        if i % 5000 == 0:
            print(f"  {i:,}개 읽음")

    cols = ["corp_code", "year", "report", "fs_div", "사전기간", "n_rows"] + KEYS + [k + "_n" for k in KEYS] + [k + "_누적칸" for k in ADD_KEYS]
    with open(OUTDIR / "계정확보_보고서별.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(per_report)

    # 기업별 집계
    agg = defaultdict(lambda: defaultdict(int))
    for s in per_report:
        a = agg[s["corp_code"]]
        a["n_all"] += 1
        a["n_pre"] += s["사전기간"]
        for k in KEYS:
            a[k + "_all"] += s[k]
            a[k + "_pre"] += s[k] and s["사전기간"]
    rows = []
    for c, f in firms.items():
        a = agg[c]
        row = {"corp_code": c, "회사명": f["회사명"], "현재_법인구분": f["현재_법인구분"], "업종2": f["업종코드"][:2],
               "연결별도_혼재": f["연결별도_혼재"], "보고서수": a["n_all"], "사전기간_보고서수": a["n_pre"]}
        for k in KEYS:
            row[k + "_전체"] = a[k + "_all"]
            row[k + "_사전"] = a[k + "_pre"]
        rows.append(row)
    with open(OUTDIR / "계정확보_기업별.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    # ── 요약 출력 ──
    N = len(per_report)
    print("\n== 1. 보고서 단위 확보율 (상태 000 응답 %d개 중 계정이 있고 값 칸이 비어 있지 않은 비율) ==" % N)
    pre = [s for s in per_report if s["사전기간"]]
    for k in KEYS:
        t = sum(s[k] for s in per_report)
        tp = sum(s[k] for s in pre)
        multi = sum(1 for s in per_report if s[k + "_n"] > 1)
        print(f"  {k:18s} 전체 {t/N:6.1%} ({t:,}) | 사전기간 {tp/len(pre):6.1%} ({tp:,}/{len(pre):,}) | 한 응답에 여러 줄 매칭 {multi:,}건")
    print("\n== 2. 누적 칸(thstrm_add_amount) 확보: 반기·3분기 응답 중 ==")
    h = [s for s in per_report if s["report"] in ("H1", "Q3")]
    for k in ADD_KEYS:
        have = [s for s in h if s[k]]
        t = sum(s[k + "_누적칸"] for s in have)
        print(f"  {k:14s} 계정이 있는 반기·3분기 응답 {len(have):,}개 중 누적 칸 있음 {t:,} ({t/max(1,len(have)):.1%})")

    print("\n== 3. 기업 단위: 사전기간 10개 보고서 중 계정이 확보된 개수 분포(기업 수) ==")
    for k in KEYS:
        cnt = Counter()
        for r in rows:
            n, tot = r[k + "_사전"], r["사전기간_보고서수"]
            cnt["0개" if n == 0 else ("보유 보고서 전부" if n == tot else "일부")] += 1
        print(f"  {k:18s} {dict(cnt)}")

    print("\n== 4. 이자 대용 계정 후보의 기업 단위 겹침: 사전기간 보유 보고서에서 한 번이라도 있는 기업 ==")
    for combo in (("interest_exp",), ("fin_cost",), ("interest_paid_cf",), ("fin_cost", "interest_paid_cf"), ("interest_exp", "fin_cost", "interest_paid_cf")):
        n_any = sum(1 for r in rows if all(r[k + "_사전"] > 0 for k in combo))
        n_full = sum(1 for r in rows if r["사전기간_보고서수"] > 0 and all(r[k + "_사전"] == r["사전기간_보고서수"] for k in combo))
        print(f"  {' + '.join(combo):45s} 한 번이라도 {n_any:,}곳 | 사전기간 보고서 전부에서 {n_full:,}곳")

    print("\n== 5. 노출도 E 분자 계정(사전기간): 한 번이라도 있는 기업 / 보유 보고서 전부에서 있는 기업 ==")
    for k in ("assets", "st_borrow", "cur_ltd", "cur_bond"):
        n_any = sum(1 for r in rows if r[k + "_사전"] > 0)
        n_full = sum(1 for r in rows if r["사전기간_보고서수"] > 0 and r[k + "_사전"] == r["사전기간_보고서수"])
        print(f"  {k:12s} {n_any:,}곳 / {n_full:,}곳")
    n_none = sum(1 for r in rows if r["st_borrow_사전"] == 0 and r["cur_ltd_사전"] == 0 and r["cur_bond_사전"] == 0)
    print(f"  차입 3계정이 사전기간에 하나도 없는 기업: {n_none:,}곳")

    print("\n== 6. 법인구분별: 영업이익·자산·(금융비용 또는 이자지급) 사전기간 전부 확보 기업 비율 ==")
    by = defaultdict(list)
    for r in rows:
        by[r["현재_법인구분"]].append(r)
    for g, rs in sorted(by.items()):
        def full(r, ks):
            return r["사전기간_보고서수"] > 0 and all(r[k + "_사전"] == r["사전기간_보고서수"] for k in ks)
        a = sum(full(r, ["op_income", "assets"]) for r in rs)
        b = sum(full(r, ["op_income", "assets", "fin_cost"]) for r in rs)
        c = sum(full(r, ["op_income", "assets", "interest_paid_cf"]) for r in rs)
        print(f"  {g}: {len(rs)}곳 | 영업이익+자산 {a/len(rs):.1%} | +금융비용 {b/len(rs):.1%} | +이자지급(CF) {c/len(rs):.1%}")

    ind = defaultdict(list)
    for r in rows:
        ind[r["업종2"]].append(r)
    ind_rows = []
    for g, rs in sorted(ind.items()):
        def full(r, ks):
            return r["사전기간_보고서수"] > 0 and all(r[k + "_사전"] == r["사전기간_보고서수"] for k in ks)
        ind_rows.append({"업종2": g, "기업수": len(rs),
                         "영업이익+자산_사전전부": sum(full(r, ["op_income", "assets"]) for r in rs),
                         "+금융비용_사전전부": sum(full(r, ["op_income", "assets", "fin_cost"]) for r in rs),
                         "+이자지급CF_사전전부": sum(full(r, ["op_income", "assets", "interest_paid_cf"]) for r in rs),
                         "차입3계정_사전_하나도없음": sum(1 for r in rs if r["st_borrow_사전"] == 0 and r["cur_ltd_사전"] == 0 and r["cur_bond_사전"] == 0)})
    with open(OUTDIR / "계정확보_업종별.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(ind_rows[0].keys()))
        w.writeheader()
        w.writerows(ind_rows)
    print("\n업종별 표는 파일로 저장했습니다:", OUTDIR / "계정확보_업종별.csv")
    print("저장:", OUTDIR)

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [7]:
main()

확인할 응답 34,964개 (기업 1,321곳)


  5,000개 읽음


  10,000개 읽음


  15,000개 읽음


  20,000개 읽음


  25,000개 읽음


  30,000개 읽음



== 1. 보고서 단위 확보율 (상태 000 응답 34964개 중 계정이 있고 값 칸이 비어 있지 않은 비율) ==
  assets             전체  99.9% (34,943) | 사전기간  99.9% (13,070/13,078) | 한 응답에 여러 줄 매칭 0건
  revenue            전체  94.6% (33,086) | 사전기간  94.8% (12,395/13,078) | 한 응답에 여러 줄 매칭 73건
  op_income          전체  97.7% (34,143) | 사전기간  97.9% (12,801/13,078) | 한 응답에 여러 줄 매칭 0건
  interest_exp       전체   4.0% (1,404) | 사전기간   4.4% (578/13,078) | 한 응답에 여러 줄 매칭 22건
  fin_cost           전체  99.0% (34,607) | 사전기간  99.1% (12,954/13,078) | 한 응답에 여러 줄 매칭 19건
  interest_paid_cf   전체  94.2% (32,946) | 사전기간  94.0% (12,298/13,078) | 한 응답에 여러 줄 매칭 289건
  st_borrow          전체  62.3% (21,795) | 사전기간  68.0% (8,889/13,078) | 한 응답에 여러 줄 매칭 25건
  cur_ltd            전체  38.7% (13,546) | 사전기간  40.7% (5,321/13,078) | 한 응답에 여러 줄 매칭 25건


  cur_bond           전체   6.3% (2,205) | 사전기간   6.2% (813/13,078) | 한 응답에 여러 줄 매칭 11건

== 2. 누적 칸(thstrm_add_amount) 확보: 반기·3분기 응답 중 ==
  op_income      계정이 있는 반기·3분기 응답 17,662개 중 누적 칸 있음 17,662 (100.0%)
  interest_exp   계정이 있는 반기·3분기 응답 709개 중 누적 칸 있음 707 (99.7%)
  fin_cost       계정이 있는 반기·3분기 응답 17,928개 중 누적 칸 있음 17,925 (100.0%)

== 3. 기업 단위: 사전기간 10개 보고서 중 계정이 확보된 개수 분포(기업 수) ==
  assets             {'보유 보고서 전부': 1314, '일부': 7}
  revenue            {'보유 보고서 전부': 1229, '일부': 40, '0개': 52}
  op_income          {'보유 보고서 전부': 1269, '일부': 36, '0개': 16}
  interest_exp       {'0개': 1240, '일부': 40, '보유 보고서 전부': 41}
  fin_cost           {'보유 보고서 전부': 1290, '일부': 25, '0개': 6}
  interest_paid_cf   {'보유 보고서 전부': 1124, '일부': 169, '0개': 28}
  st_borrow          {'보유 보고서 전부': 744, '일부': 278, '0개': 299}
  cur_ltd            {'일부': 329, '보유 보고서 전부': 365, '0개': 627}
  cur_bond           {'0개': 1183, '일부': 109, '보유 보고서 전부': 29}

== 4. 이자 대용 계정 후보의 기업 단위 겹침: 사전기간 보유 보고서에서 한 번이라도 있는 기업 ==
  interest_exp

### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [8]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/05_계정확보/계정확보_기업별.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

계정확보_기업별.csv | 1,321행 × 25열


,corp_code,회사명,현재_법인구분,업종2,연결별도_혼재,보고서수,사전기간_보고서수,assets_전체,assets_사전,revenue_전체,...,fin_cost_전체,fin_cost_사전,interest_paid_cf_전체,interest_paid_cf_사전,st_borrow_전체,st_borrow_사전,cur_ltd_전체,cur_ltd_사전,cur_bond_전체,cur_bond_사전
0,164618,현대사료,K,10,True,26,10,26,10,26,...,26,10,20,10,23,10,9,3,0,0
1,1046708,바디텍메드,K,21,False,27,10,27,10,27,...,27,10,27,10,26,10,24,10,0,0
2,493431,자안바이오,E,20,False,14,10,14,10,14,...,14,10,14,10,14,10,3,3,1,1
3,608839,에이루트,K,26,False,27,10,27,10,27,...,27,10,27,10,16,6,13,5,0,0
4,869272,서전기전,K,28,False,27,10,27,10,26,...,27,10,26,9,20,8,0,0,0,0
